# How to view previous jobs and retrieve results

**Viewing metadata and retrieving results from jobs previously submitted to Fire Opal**

Fire Opal allows you to view a list of previous jobs with metadata and to retrieve results from previously submitted jobs.

**_Note: To achieve the highest degree of accuracy, be sure to always retrieve results directly from Fire Opal. Results obtained from hardware providers' platforms and SDKs will not include Fire Opal's post-processing will therefore be of lower quality._** 

In this guide, you will learn how to:

1. Retrieve results or status from a FireOpalJob object.
1. View and filter a list of previous jobs.
1. Use a job's unique ID to retrieve the results.
1. Get a list of job metadata.

### 1. Poll for results or get status from a FireOpalJob object

Fire Opal's job execution functions (`execute`, `iterate`, and `solve_qaoa`) work asynchronously and return a `FireOpalJob` object. You can use this job object to poll for results synchronously while the job is processing, query the job's status, and retrieve results once the job has completed.

In [10]:
import fireopal as fo

In [ ]:
# When using a browser-based development platform, you must authenticate using an API key.
api_key = "YOUR_QCTRL_API_KEY"
fo.authenticate_qctrl_account(api_key=api_key)

In [22]:
# Run a circuit execution job
fire_opal_job = fo.execute(...)

You can use this `FireOpalJob` object to query the job's status.

In [31]:
print(fire_opal_job.status())

{'status_message': 'Job has finished successfully.', 'action_status': 'SUCCESS'}


You can use `.result()` to poll for the job results if the job is still processing or to retrieve results once the job has finished.

In [33]:
# Poll for results synchronously
fire_opal_results = fire_opal_job.result()

### 2. View list of previous jobs and metadata

To view pre-formatted metadata, use the `activity_monitor` function. As shown below, this function's default behavior is to display metadata from the most recent job.

In [34]:
from fireopal import activity_monitor

In [35]:
activity_monitor()

Getting jobs for all statuses. To filter jobs by status,  use the status keyword argument. Valid status values are: SUCCESS, FAILURE, REVOKED, PENDING, RECEIVED, RETRY, STARTED.

| Function | Status  | Created at (UTC)    | Updated at (UTC)    | Action ID |
| -------- | ------- | ------------------- | ------------------- | --------- |
| execute  | SUCCESS | 2025-02-10 23:56:42 | 2025-02-10 23:56:45 | 185       |


To view metadata for more than one job, use the `limit` parameter. For example, the code below displays the five most recent jobs.

In [36]:
activity_monitor(limit=5)

Getting jobs for all statuses. To filter jobs by status,  use the status keyword argument. Valid status values are: SUCCESS, FAILURE, REVOKED, PENDING, RECEIVED, RETRY, STARTED.

| Function   | Status  | Created at (UTC)    | Updated at (UTC)    | Action ID |
| ---------- | ------- | ------------------- | ------------------- | --------- |
| execute    | SUCCESS | 2025-02-10 23:56:42 | 2025-02-10 23:56:45 | 185       |
| execute    | SUCCESS | 2025-02-10 23:56:41 | 2025-02-10 23:56:45 | 184       |
| execute    | SUCCESS | 2025-02-10 23:56:39 | 2025-02-10 23:56:40 | 183       |
| execute    | SUCCESS | 2025-02-10 23:55:35 | 2025-02-10 23:55:42 | 182       |
| solve_qaoa | SUCCESS | 2025-02-10 21:47:53 | 2025-02-10 21:48:19 | 181       |


To only view older jobs, the `offset` parameter can be used to skip more recent jobs. For example, the code below skips the five most recent jobs and views the sixth most recent.

In [37]:
activity_monitor(limit=1, offset=5)

Getting jobs for all statuses. To filter jobs by status,  use the status keyword argument. Valid status values are: SUCCESS, FAILURE, REVOKED, PENDING, RECEIVED, RETRY, STARTED.

| Function | Status  | Created at (UTC)    | Updated at (UTC)    | Action ID |
| -------- | ------- | ------------------- | ------------------- | --------- |
| iterate  | SUCCESS | 2025-02-10 21:40:28 | 2025-02-10 21:40:30 | 180       |


As the outputs above indicate, we can also filter by status.

In [38]:
activity_monitor(limit=4, status="SUCCESS")

| Function | Status  | Created at (UTC)    | Updated at (UTC)    | Action ID |
| -------- | ------- | ------------------- | ------------------- | --------- |
| execute  | SUCCESS | 2025-02-10 23:56:42 | 2025-02-10 23:56:45 | 185       |
| execute  | SUCCESS | 2025-02-10 23:56:41 | 2025-02-10 23:56:45 | 184       |
| execute  | SUCCESS | 2025-02-10 23:56:39 | 2025-02-10 23:56:40 | 183       |
| execute  | SUCCESS | 2025-02-10 23:55:35 | 2025-02-10 23:55:42 | 182       |


### 3. Use a job's Action ID to retrieve results

The results of a successfully completed job can be retrieved by passing its Action ID to the `get_result` function.

**_Note: This method will poll until the job has fully completed. Even in the case of kernel disconnection, the job will still complete, and results can later be retrieved._** 

In [39]:
import warnings

action_id = "YOUR_JOBS_ACTION_ID"

# The get_result function will display any runtime warnings
# captured during the function call that generated the job.
# Warnings are ignored in the following example for brevity.
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    result = fo.get_result(action_id)

print(result)

{'results': [{'00': 0.5146484375, '11': 0.4853515625}], 'provider_job_ids': ['daea76b3-96ff-4b26-8c5f-5be3c0f97114']}


### 4. Get related hardware provider job IDs
Fire Opal applies error suppression and submits optimizes circuits to your hardware provider and backend of choice. The associated job IDs of jobs submitted to the hardware provider platform can be retrieved in the job results dictionary under the key "provider_job_ids".


In [40]:
provider_job_ids = result["provider_job_ids"]
print(provider_job_ids)

['daea76b3-96ff-4b26-8c5f-5be3c0f97114']


### 5. Get a list of job metadata

Finally, note that it's possible to instead retrieve information (metadata) about the job, such as the job status or time of creation. To do so, use the `get_action_metadata` function, which supports the same parameters as the `activity_monitor` and returns a list, as shown in the following example.

In [44]:
metadata = fo.get_action_metadata(limit=4, status="SUCCESS")

# Each list element is an ActionMetadata instance, which is just a dataclass.
print(metadata[0])
print("\n")

# Like any dataclass, we can access the attribute values through dot notation.
print(
    f"Function name: {metadata[0].name}\n"
    f"Job status: {metadata[0].status}\n"
    f"Job created at: {metadata[0].created_at}\n"
    f"Job updated at: {metadata[0].updated_at}\n"
    f"Action ID: {metadata[0].model_id}"
)

ActionMetadata(name='execute', status='SUCCESS', created_at='2025-02-10 23:56:42', updated_at='2025-02-10 23:56:45', model_id='185')


Function name: execute
Job status: SUCCESS
Job created at: 2025-02-10 23:56:42
Job updated at: 2025-02-10 23:56:45
Action ID: 185


Now that you know how to view previous jobs and retrieve results, try [creating and running circuits](https://docs.q-ctrl.com/fire-opal/tutorials/creating-and-running-circuits) and you'll be able to retrieve the results at your convenience.

In [45]:
from fireopal import print_package_versions

print_package_versions()

| Package               | Version |
| --------------------- | ------- |
| Python                | 3.11.9  |
| networkx              | 2.8.8   |
| numpy                 | 1.26.4  |
| qiskit                | 1.1.0   |
| sympy                 | 1.13.3  |
| fire-opal             | 8.2.1   |
| qctrl-workflow-client | 5.1.2   |
